## Audit Bronze 作成ハンズオン（dirty CSV版）

このノートブックは `audit_dirty.csv` を Bronze テーブル化する演習です。

進め方:
- まずは **All Run** して、dirty CSV の影響（カラムずれ・JSON不整合）が残ることを確認
- その後、**AIチャット（Genie）** に相談しながら加工ロジックを改善
- audit では「加工ロジック以外」は提供済みです（usageで実装量を増やします）


In [0]:
%run ../config


### 1) Bronzeテーブル定義（先に作成）
- Bronzeは「生データを保持する層」なので、基本はSTRING中心
- 監査列 `_datasource`, `_ingest_timestamp` を追加


In [0]:
BRONZE_AUDIT_SCHEMA_SQL = """
    `event_id` STRING,
    `event_time` STRING,
    `action_name` STRING,
    `user` STRING,
    `request_params` STRING,
    `resource_name` STRING,
    `source_ip` STRING
"""

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_audit_table_path} (
    {BRONZE_AUDIT_SCHEMA_SQL},
    _datasource STRING,
    _ingest_timestamp TIMESTAMP
)
USING DELTA
""")

print("created/exists:", bronze_audit_table_path)


### 2) まずは加工なしで読み込む（All Run で現象確認）
ここでは意図的に最小オプションで読み込み、データ由来の問題を観察します。


In [0]:
from pyspark.sql import functions as F

df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(audit_csv_path)
)

display(df)
print("row count:", df.count())


### カラムのずれを解消

userカラム値にあるカンマにより、userカラムのデータが分割されています。  
例：`"{""email"": ""user000@example.com""` **,** `""name"": ""佐藤 美咲""}"`  
ダブルクォートで一つに括り、カンマをエスケープしましょう。  

**AIチャット（Genie）で加工ロジックを作る**

次セルを、以下のようなプロンプトで改善してください。

プロンプト例:
```text
userカラム値にあるカンマにより、userカラムのデータが分割されています。  
bronze_ready_dfにCSVファイルを取り込む際に、ダブルクォートで括られた値を一つのカラム値にして。
```


In [0]:
# TODO: 以下コメントを外し、ここを受講者が改善する
# df = (
#     spark.read.format("csv")
#     .option()
#     .load(audit_csv_path)
# )

print("bronze_ready row count:", bronze_ready_df.count())
display(bronze_ready_df.limit(20))

In [0]:
# 監査列として`_datasource`列と`_ingest_timestamp`列を追加
bronze_ready_df = (
    bronze_ready_df.select("*", "_metadata")
    .withColumn("_datasource", df["_metadata.file_path"])
    .withColumn(
        "_ingest_timestamp",
        from_utc_timestamp(col("_metadata.file_modification_time"), "Asia/Tokyo"),
    )
    .drop("_metadata")
)

### 5) Bronzeテーブルへ書き込み


In [0]:
# (
#     bronze_ready_df.write.format("delta")
#     .mode("append")
#     .saveAsTable(bronze_audit_table_path)
# )

# print("saved:", bronze_audit_table_path)


In [0]:
display(spark.table(bronze_audit_table_path).orderBy(F.col("_ingest_timestamp").desc()).limit(50))
